In [21]:
import sys
sys.path.insert(0, '..')
from dependencies import *

subject_model_predictors = eelbrain.load.unpickle(PREDICTOR_DIR / f'~concatenated_predictors.pickle')
models = get_models(exclude=['envelope_log_onset'])

In [ ]:
# -----------------------------
# Storage
# -----------------------------
r_values = {model: [] for model in models}
r2_values = {model: [] for model in models}

# -----------------------------
# Loop over subjects
# -----------------------------
for subject in SUBJECTS:
    print(f'\n===== Subject {subject} =====')

    eeg = eelbrain.load.unpickle(EEG_DIR / subject / f'{subject}concatenated_eeg.pickle')
    eeg_data = eeg.x  # (channels, time)

    for model in models:
        print(f'\nModel: {model}')

        # Load subject TRF
        trf_subject = eelbrain.load.unpickle(TRF_DIR / subject / f'{subject} {model}.pickle')
        predictors = subject_model_predictors[subject][model]

        # Build stimulus (subject predictors)
        stimulus = predictors

        # Predict EEG
        predicted_subject = eelbrain.convolve(trf_subject.h_scaled, stimulus)
        pred = predicted_subject.x

        # Per-channel correlation
        r_channels = []
        for ch in range(eeg_data.shape[0]):
            # Skip channels with zero variance
            if np.std(eeg_data[ch]) > 0 and np.std(pred[ch]) > 0:
                r_channels.append(np.corrcoef(eeg_data[ch], pred[ch])[0,1])

        # Skip TRF if all channels are invalid
        if len(r_channels) == 0:
            print("All channels invalid for this TRF, skipping.")
            continue

        # Mean across valid channels
        r_mean = np.mean(r_channels)
        r2_mean = r_mean**2

        r_values[model].append(r_mean)
        r2_values[model].append(r2_mean)

        print(f'Subject TRF: r = {r_mean:.4f}, r² = {r2_mean:.4f}')

# -----------------------------
# Summary statistics: Subject TRFs
# -----------------------------
print('\n===== Subject TRF summary =====')
for model in models:
    if len(r_values[model]) == 0:
        print(f'{model}: No valid TRFs!')
        continue
    mean_r = np.mean(r_values[model])
    std_r = np.std(r_values[model])
    mean_r2 = np.mean(r2_values[model])
    std_r2 = np.std(r2_values[model])
    print(f'{model}: r = {mean_r:.4f} ± {std_r:.4f}, r² = {mean_r2:.4f} ± {std_r2:.4f}')



===== Subject S05 =====

Model: envelope_log
Subject TRF: r = 0.0367, r² = 0.0013

Model: envelope_onset
Subject TRF: r = 0.0369, r² = 0.0014

===== Subject S34 =====

Model: envelope_log
Subject TRF: r = 0.0616, r² = 0.0038

Model: envelope_onset
Subject TRF: r = 0.0435, r² = 0.0019

===== Subject S35 =====

Model: envelope_log
Subject TRF: r = 0.0702, r² = 0.0049

Model: envelope_onset
Subject TRF: r = 0.0427, r² = 0.0018

===== Subject S03 =====

Model: envelope_log
Subject TRF: r = 0.0620, r² = 0.0038

Model: envelope_onset
Subject TRF: r = 0.0233, r² = 0.0005

===== Subject S04 =====

Model: envelope_log
Subject TRF: r = 0.0512, r² = 0.0026

Model: envelope_onset
Subject TRF: r = 0.0368, r² = 0.0014

===== Subject S44 =====

Model: envelope_log
Subject TRF: r = 0.0098, r² = 0.0001

Model: envelope_onset
Subject TRF: r = 0.0082, r² = 0.0001

===== Subject S26 =====

Model: envelope_log
Subject TRF: r = -0.0014, r² = 0.0000

Model: envelope_onset
Subject TRF: r = -0.0004, r² = 0.00

In [3]:
t_stat, p_val = ttest_rel(r_values['envelope_log'], r_values['envelope_onset'])
print(f'Envelope log vs. onset: t={t_stat:.2f}, p={p_val:.4f}')

Envelope log vs. onset: t=8.52, p=0.0000


In [ ]:
# ------------------------------------------------
# Storage
# ------------------------------------------------
r_values_universal = {model: [] for model in models}
r2_values_universal = {model: [] for model in models}

# ------------------------------------------------
# Loop over models
# ------------------------------------------------
for model in models:

    print(f'\n===== Universal encoder: {model} =====')

    # ----------------------------------------
    # Load universal TRF (forward model)
    # ----------------------------------------
    trf_universal = eelbrain.load.unpickle(
        TRF_DIR / f'universal-trf-{model}.pickle'
    )

    # ----------------------------------------
    # Loop over subjects
    # ----------------------------------------
    for subject in SUBJECTS:

        print(f'\nSubject: {subject}')

        # Load EEG
        eeg = eelbrain.load.unpickle(
            EEG_DIR / subject / f'{subject}concatenated_eeg.pickle'
        )
        eeg_data = eeg.x  # (channels, time)

        # Get predictors
        predictors = subject_model_predictors[subject][model]

        # ----------------------------------------
        # Build stimulus (IMPORTANT)
        # ----------------------------------------
        stimulus = predictors[0]  # or sum(predictors) if needed

        # ----------------------------------------
        # Predict EEG (FORWARD MODEL)
        # ----------------------------------------
        predicted = eelbrain.convolve(trf_universal, stimulus)
        pred_data = predicted.x

        # ----------------------------------------
        # Per-channel correlation
        # ----------------------------------------
        r_channels = []
        for ch in range(eeg_data.shape[0]):
            if np.std(eeg_data[ch]) > 0 and np.std(pred_data[ch]) > 0:
                r_channels.append(
                    np.corrcoef(eeg_data[ch], pred_data[ch])[0, 1]
                )

        # Skip if invalid
        if len(r_channels) == 0:
            print("All channels invalid, skipping.")
            continue

        # ----------------------------------------
        # Aggregate
        # ----------------------------------------
        r_mean = np.mean(r_channels)
        r2_mean = r_mean ** 2  # (same as your subject block)

        r_values_universal[model].append(r_mean)
        r2_values_universal[model].append(r2_mean)

        print(f'r = {r_mean:.4f}, r² = {r2_mean:.4f}')

# ------------------------------------------------
# Summary statistics
# ------------------------------------------------
print('\n===== Universal encoder summary =====')

for model in models:
    mean_r = np.mean(r_values_universal[model])
    std_r = np.std(r_values_universal[model])
    mean_r2 = np.mean(r2_values_universal[model])
    std_r2 = np.std(r2_values_universal[model])

    print(f'{model}: r = {mean_r:.4f} ± {std_r:.4f}, r² = {mean_r2:.4f} ± {std_r2:.4f}')



===== Universal encoder: envelope_log =====

Subject: S05
r = 0.0111, r² = 0.0001

Subject: S34
r = 0.0321, r² = 0.0010

Subject: S35
r = 0.0346, r² = 0.0012

Subject: S03
r = 0.0418, r² = 0.0017

Subject: S04
r = 0.0364, r² = 0.0013

Subject: S44
r = 0.0157, r² = 0.0002

Subject: S26
r = 0.0014, r² = 0.0000

Subject: S19
r = 0.0422, r² = 0.0018

Subject: S21
r = 0.0515, r² = 0.0027

Subject: S17
r = 0.0522, r² = 0.0027

Subject: S10
r = 0.0097, r² = 0.0001

Subject: S42
r = 0.0150, r² = 0.0002

Subject: S45
r = 0.0069, r² = 0.0000

Subject: S11
r = 0.0206, r² = 0.0004

Subject: S16
r = 0.0743, r² = 0.0055

Subject: S20
r = 0.0594, r² = 0.0035

Subject: S18
r = 0.0497, r² = 0.0025

Subject: S01
r = 0.0768, r² = 0.0059

Subject: S39
r = 0.0221, r² = 0.0005

Subject: S06
r = 0.0586, r² = 0.0034

Subject: S08
r = 0.0406, r² = 0.0016

Subject: S37
r = 0.0171, r² = 0.0003

Subject: S36
r = 0.0389, r² = 0.0015

Subject: S38
r = 0.0077, r² = 0.0001

Subject: S22
r = 0.0083, r² = 0.0001

Subj